In [ ]:
import os
import time
import math
import json
import random
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

### Excess word list with frequency cutoff (lemmatized)
To determine the optimal rare word list that yields the best LLM use estimate,
we try different frequency cutoffs. The base list consists of all excess style 
words from the 2024 pubmed data. For each cutoff value, only words with frequency 
lower than the cutoff (average 2024) are considered into the list 
of rare words. 

Procedure:
- get all lemmatized 2024 excess words from the previous paper (need them lemmatized bc my frequencies are lemmatized)
- filter yearly freqs to only include words with frequency > 1e-3 to remove noise (e.g. "aaaa") (That's not really necessary though, bc we only look at words present in the excess word list anyways -> remove this step)
- filter list to only include words appearing in my yearly frequencies for 2024
- filter list to only include words with average 2024 frequency below cutoff
- get raw count matrix and group counts for all words in list
- PROBLEM: the frequencies are based on all lemmatized versions, so the list needs to include all original words that map to the same lemma to get accurate frequencies
- use new counts to compute frequency, diffs, ratios and projection 
- repeat for every cutoff and section


In [ ]:
VERSION = "research-article_aimrd_f_run2"
RESULTS_PATH = "../data/results/baseline_2025-06-26/" + VERSION
secs = next(os.walk(RESULTS_PATH))[1]

In [ ]:
years = np.arange(2000, 2026)
months = np.arange(1, 13)

In [ ]:
freqs_2024 = {}
for sec in secs:
    freqs, _, _ = utils.get_frequency_projection_yearly(RESULTS_PATH, sec)
    freqs = freqs["2024"]
    # try results with or without this, it's not immediately clear to me which version is better
    # freqs = freqs[freqs > 1e-3]
    freqs_2024[sec] = freqs

In [ ]:
freqs_2024["introduction"].loc["discern"]

In [ ]:
excess_words = {}
words = {}
all_lemmas = {}
for sec in secs:
    excess = [
        w
        for w in utils.excess_style_words_2024_lemmatized
        if w in freqs_2024[sec].index
    ]
    print(
        f"{sec}: {len(excess)}/{len(utils.excess_style_words_2024_lemmatized)} excess words"
    )
    excess_words[sec] = excess

    # de-lemmatize (get lemmas before cutoff filtering, append de-lemmatized words
    # to excess words after cutoff filtering)
    w = np.load(
        os.path.join(RESULTS_PATH, sec, f"words_{sec}.pkl.npy"), allow_pickle=True
    )
    words[sec] = w
    all_lemmas[sec] = utils.get_lemma_dict(w)

In [ ]:
l = utils.get_lemma_dict(utils.excess_style_words_2024_lemmatized)
l

In [ ]:
sec = "introduction"
[w for w in utils.excess_style_words_2024_lemmatized if not w in freqs_2024[sec].index]

In [ ]:
cutoffs = [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
excess_words_cutoff = {}
cutoff_counts_before = np.zeros((len(secs), len(cutoffs)), dtype=int)
cutoff_counts_after = np.zeros((len(secs), len(cutoffs)), dtype=int)

for i, sec in enumerate(secs):
    excess_words_cutoff[sec] = {}
    for j, cutoff in enumerate(cutoffs):
        excess_words_c = list(
            freqs_2024[sec]
            .loc[excess_words[sec]][freqs_2024[sec].loc[excess_words[sec]] < cutoff]
            .index
        )
        cutoff_counts_before[i, j] = len(excess_words_c)

        for word, lemma in all_lemmas[sec].items():
            if lemma in excess_words_c:
                excess_words_c.append(word)

        cutoff_counts_after[i, j] = len(excess_words_c)
        excess_words_cutoff[sec][cutoff] = excess_words_c

pd.DataFrame(cutoff_counts_before, columns=cutoffs, index=secs)

In [ ]:
pd.DataFrame(cutoff_counts_after, columns=cutoffs, index=secs)

In [ ]:
len(utils.chatgptwords_rare)

In [ ]:
freqs_dfs = {}

for sec in secs:
    freqs_dfs[sec] = utils.get_cutoff_frequency(
        RESULTS_PATH, sec, excess_words_cutoff[sec]
    )

In [ ]:
# filter out cutoffs with too high base frequency
for sec in secs:
    remove = freqs_dfs[sec][freqs_dfs[sec]["projection"] >= 0.95]["cutoff"].unique()
    freqs_dfs[sec] = freqs_dfs[sec][
        freqs_dfs[sec]["cutoff"].apply(lambda x: x not in remove)
    ]

In [ ]:
freqs_dfs["abstract"]

In [ ]:
freqs_dfs["introduction"]

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(5, 5), layout="constrained")

hue_norm = LogNorm()
palette = "flare"
data = freqs_dfs["introduction"]
for i, var in enumerate(["frequency", "diff", "usage estimate"]):
    ax = axs.flatten()[i]

    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=data,
        x="time",
        y=var,
        hue="cutoff",
        hue_norm=hue_norm,
        palette=palette,
        ax=ax,
        legend=False,
    )
    """
    handles = [Line2D([], []) for _ in cutoffs]
    cmap = plt.get_cmap(palette)
    for h, key in zip(handles, cutoffs):
        h.set_linestyle("-")
        h.set_color(cmap(hue_norm(key)))
        h.set_label(f"{key}")
    ax.legend(handles=handles, title="cutoff")
    """
    if var == "frequency":
        sns.lineplot(
            data=data,
            x="time",
            y="projection",
            hue="cutoff",
            hue_norm=LogNorm(),
            palette=palette,
            ax=ax,
            alpha=0.5,
            linestyle="-.",
            legend=False,
        )
    if var == "usage estimate":
        ax.set_ylim(-0.01, 1.01)

    sm = plt.cm.ScalarMappable(cmap=palette, norm=hue_norm)
    cax = fig.add_axes(
        [
            ax.get_position().x1 + 0.1,
            ax.get_position().y0,
            0.01,
            ax.get_position().height,
        ]
    )
    ax.figure.colorbar(sm, cax=cax)

In [ ]:
freqs_df_agg = {}
for sec in secs:
    df = freqs_dfs[sec].copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "cutoff"]).mean().reset_index()
    freqs_df_agg[sec] = df

In [ ]:
freqs_df_agg["introduction"][freqs_df_agg["introduction"]["time"] == 2024]

In [ ]:
f = freqs_df_agg["introduction"][freqs_df_agg["introduction"]["time"] == 2024]

fig = plt.figure(figsize=(3.5, 2), layout="constrained")
plt.subplot(121)
plt.plot(f["cutoff"], f["frequency"], ".-", clip_on=False, label="Observed\nfrequency")
plt.plot(f["cutoff"], f["projection"], ".-", clip_on=False, label="Expected\nfrequency")
plt.legend()
plt.xscale("log")
plt.xlim([1e-3, 1])
plt.ylim([0, 1])
plt.ylabel("Frequency")
plt.xlabel("Threshold")

plt.subplot(122)
plt.plot(f["cutoff"], f["diff"], "k.-", label="Frequency gap (Delta)")
plt.plot(f["cutoff"], f["usage estimate"], "k.-", label="Frequency gap (Delta)")
plt.xscale("log")
plt.xlim([1e-3, 1.1])
# plt.ylim([0, 0.14])
plt.ylabel(r"Frequency gap ($\Delta$)")
plt.xlabel("Threshold")

In [ ]:
# ensure correct order
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]

fig = plt.figure(figsize=(7, 6), layout="constrained")
subfigs = fig.subfigures(3, 2)  # outer grid

for subfig, sec in zip(subfigs.flat, sections):
    df = freqs_df_agg[sec]
    f = df[df["time"] == 2025]

    axs = subfig.subplots(1, 2)  # inner grid

    subfig.suptitle(sec)

    # frequency plot
    axs[0].plot(
        f["cutoff"], f["frequency"], ".-", clip_on=False, label="Observed\nfrequency"
    )
    axs[0].plot(
        f["cutoff"], f["projection"], ".-", clip_on=False, label="Expected\nfrequency"
    )
    axs[0].set_xscale("log")
    axs[0].set_xlim([1e-3, 1])
    axs[0].set_ylim([0, 1])
    axs[0].set_ylabel("Frequency")
    axs[0].set_xlabel("Threshold")
    if sec == "abstract":
        axs[0].legend()

    # usage estimate plot
    axs[1].plot(f["cutoff"], f["diff"], "k.-", label="Δ")
    axs[1].plot(f["cutoff"], f["usage estimate"], "r.-", label="Δ / (1-Proj)")

    i = int(np.argmax(f["usage estimate"]))
    axs[1].text(
        list(f["cutoff"])[i],
        list(f["usage estimate"])[i] + 0.03,
        f"{list(f["usage estimate"])[i]:.2f}",
        va="center",
        bbox=dict(
            edgecolor="none", facecolor="#666666", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )
    i = int(np.argmax(f["diff"]))
    axs[1].text(
        list(f["cutoff"])[i],
        list(f["diff"])[i] + 0.03,
        f"{list(f["diff"])[i]:.2f}",
        va="center",
        bbox=dict(
            edgecolor="none", facecolor="#666666", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )

    axs[1].set_xscale("log")
    axs[1].set_xlim([1e-3, 1.15])
    axs[1].set_ylim([0, 0.7])
    axs[1].set_ylabel(r"Frequency gap ($\Delta$)")
    axs[1].set_xlabel("Threshold")
    if sec == "abstract":
        axs[1].legend()

plt.savefig(
    os.path.join("../results/plots", VERSION, "lower_bound_lemmatized.png"), dpi=300
)

In [ ]:
excess_words_cutoff["introduction"][0.05]